# EP16 — RLHF & InstructGPT
**COMPSCI 713 · S1 2025 Q7 · 1 mark**

**Question:** Which MCQ option correctly maps RL/MDP components to the RLHF framework?

**Answer: C**

---

## Why C is Correct

| MDP Component | RLHF Mapping | Reason |
|--------------|-------------|--------|
| **State** | User prompt | The current situation the agent (LM) observes |
| **Action** | Model-generated response | The output the agent produces in response to the state |
| **Policy** | Language model | The model that maps states (prompts) to actions (responses) |
| **Reward** | Score from reward model trained on human preferences | The signal telling the policy how good its action was |

---

## Why the Others are Wrong

**A is wrong:** Saying the action is "human preference" confuses the data collection phase (Step 2) with the RL training phase (Step 3). Human preferences are used to *train the reward model*, not as actions in the MDP.

**B is wrong:** A token is not a meaningful *action* in the RLHF MDP — the full *response* is the action. And the reward model scores responses, it is not the policy.

**D is wrong:** An "assigned rank" is not an action the agent takes; it's an evaluation of the action. And "accuracy of ranking vs human labels" describes reward model training accuracy, not the RL reward signal.

---

## The Three RLHF Steps (InstructGPT)

### Step 1 — Supervised Fine-Tuning (SFT)
- Human labellers write ideal responses to prompts
- GPT-3 is fine-tuned on these demonstrations (supervised learning)
- Result: a decent but not yet aligned model

### Step 2 — Reward Model Training
- A prompt is shown and the SFT model generates multiple outputs
- Human labellers *rank* the outputs (best → worst)
- A separate reward model is trained to predict these rankings
- The reward model learns to score any (prompt, response) pair

### Step 3 — RL with PPO
- The SFT model is used as starting policy
- For each new prompt (state), the policy generates a response (action)
- The reward model scores the response (reward)
- PPO updates the policy to maximise reward
- A KL penalty prevents the policy drifting too far from SFT (avoids reward hacking)

---

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

# =============================================================
# Visualise the 3-step RLHF pipeline
# =============================================================

fig, axes = plt.subplots(1, 3, figsize=(16, 6))

steps = [
    {
        'title': 'Step 1: Supervised Fine-Tuning',
        'colour': '#3498db',
        'boxes': [
            ('Prompt dataset', '#d6eaf8'),
            ('Human writes\nideal response', '#aed6f1'),
            ('SFT on GPT-3\n(cross-entropy loss)', '#7fb3d3'),
            ('SFT Model', '#2980b9'),
        ],
        'note': 'Expensive but gives\na good starting policy'
    },
    {
        'title': 'Step 2: Reward Model Training',
        'colour': '#e67e22',
        'boxes': [
            ('Prompt → SFT model\ngenerates 4-9 outputs', '#fdebd0'),
            ('Human ranks outputs\nbest → worst', '#f0b27a'),
            ('Train reward model\nto predict rankings', '#e59866'),
            ('Reward Model (RM)', '#ca6f1e'),
        ],
        'note': 'RM learns to score\nany (prompt, response)'
    },
    {
        'title': 'Step 3: PPO Reinforcement Learning',
        'colour': '#27ae60',
        'boxes': [
            ('New prompt\n(STATE)', '#d5f5e3'),
            ('Policy (LM)\ngenerates response (ACTION)', '#a9dfbf'),
            ('RM scores response\n(REWARD) + KL penalty', '#7dcea0'),
            ('PPO updates policy\n→ aligned LM', '#1e8449'),
        ],
        'note': 'Maps cleanly to MDP:\nstate→action→reward'
    },
]

for ax, step in zip(axes, steps):
    ax.axis('off')
    ax.set_xlim(0, 10)
    ax.set_ylim(0, 12)
    ax.set_title(step['title'], fontsize=10, fontweight='bold',
                 color=step['colour'], pad=8)

    y_positions = [9.5, 7.0, 4.5, 2.0]
    for i, ((label, box_colour), y) in enumerate(zip(step['boxes'], y_positions)):
        ax.add_patch(plt.Rectangle((1, y), 8, 1.7, color=box_colour,
                                    ec=step['colour'], lw=1.5, zorder=2))
        ax.text(5, y + 0.85, label, ha='center', va='center',
                fontsize=9, zorder=3, color='#1a1a1a')
        if i < 3:
            ax.annotate('', xy=(5, y_positions[i+1] + 1.7),
                        xytext=(5, y),
                        arrowprops=dict(arrowstyle='->', color=step['colour'], lw=2))

    # Note at bottom
    ax.text(5, 0.5, step['note'], ha='center', va='center',
            fontsize=8.5, color='#555', style='italic')

# MDP mapping annotation on step 3
axes[2].text(9.5, 9.0, 'S', fontsize=13, color='#c0392b', fontweight='bold')
axes[2].text(9.5, 6.5, 'A', fontsize=13, color='#8e44ad', fontweight='bold')
axes[2].text(9.5, 4.0, 'R', fontsize=13, color='#e67e22', fontweight='bold')
axes[2].text(9.5, 1.5, 'π', fontsize=13, color='#27ae60', fontweight='bold')

fig.suptitle('RLHF Pipeline (InstructGPT)\nAnswer C: State=prompt, Action=response, Policy=LM, Reward=RM score',
             fontsize=12, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================
# Simulate: reward model training on ranked outputs
# Shows how human rankings become a reward signal
# =============================================================

np.random.seed(0)

# Simulate 100 (prompt, response) pairs with human-assigned scores
# Features: [helpfulness_signal, harmlessness_signal, coherence_signal]
n_samples = 200

# True reward function (unknown to model): weighted combination
true_weights = np.array([0.5, 0.3, 0.2])  # help, harmless, coherent

X = np.random.randn(n_samples, 3)  # response feature vectors
y = X @ true_weights + np.random.randn(n_samples) * 0.1  # noisy human scores

# Sort to simulate human ranking
ranking_order = np.argsort(-y)  # human ranks best first

# Train a simple linear reward model
from numpy.linalg import lstsq
learned_weights, _, _, _ = lstsq(X, y, rcond=None)

# Evaluate: does the reward model agree with human rankings?
y_pred = X @ learned_weights
spearman_r = np.corrcoef(y, y_pred)[0, 1]

print('=== Reward Model Training Simulation ===')
print(f'True weights (human preference): {true_weights}')
print(f'Learned weights (reward model):  {np.round(learned_weights, 3)}')
print(f'Correlation with human scores:   {spearman_r:.4f}')
print()
print('Top 5 responses by human ranking:')
for i, idx in enumerate(ranking_order[:5]):
    print(f'  Rank {i+1}: human_score={y[idx]:.3f}, RM_score={y_pred[idx]:.3f}')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Left: RM scores vs human scores
ax = axes[0]
ax.scatter(y, y_pred, alpha=0.5, color='#3498db', s=20)
ax.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=1.5)
ax.set_xlabel('Human preference score')
ax.set_ylabel('Reward model predicted score')
ax.set_title(f'RM vs Human Scores (r={spearman_r:.3f})')
ax.grid(True, alpha=0.3)

# Right: visualise weights
ax = axes[1]
x = np.arange(3)
width = 0.35
ax.bar(x - width/2, true_weights, width, label='True human pref.', color='#2ecc71', alpha=0.8)
ax.bar(x + width/2, learned_weights, width, label='Learned RM weights', color='#e74c3c', alpha=0.8)
ax.set_xticks(x)
ax.set_xticklabels(['Helpfulness', 'Harmlessness', 'Coherence'])
ax.set_ylabel('Weight')
ax.set_title('Reward Model Learns Human Preference Weights')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3, axis='y')

plt.suptitle('Step 2: Reward Model Training from Human Rankings', fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# =============================================================
# Simulate PPO-style policy improvement (simplified)
# Shows policy moving toward higher reward responses over time
# =============================================================

np.random.seed(7)

# Policy is a distribution over 5 response 'types'
# Each type has a fixed reward (from trained RM)
response_rewards = np.array([0.2, 0.5, 0.8, 0.4, 0.9])  # type 4 is best
response_names   = ['Refuse', 'Vague', 'Helpful', 'Verbose', 'Helpful+Concise']

# Start: uniform policy (SFT baseline)
policy_logits = np.zeros(5)
kl_weight     = 0.1  # KL penalty to prevent policy drifting too far
sft_logits    = np.zeros(5)  # reference SFT policy

hist_probs   = []
hist_rewards = []

for step in range(200):
    # Softmax policy
    probs  = np.exp(policy_logits) / np.exp(policy_logits).sum()
    # Expected reward
    exp_r  = probs @ response_rewards
    # KL divergence from SFT
    sft_probs = np.exp(sft_logits) / np.exp(sft_logits).sum()
    kl        = np.sum(probs * np.log(probs / sft_probs + 1e-9))
    # PPO-style gradient ascent on reward minus KL penalty
    grad = response_rewards - kl_weight * np.log(probs / sft_probs + 1e-9)
    policy_logits += 0.05 * grad

    hist_probs.append(probs.copy())
    hist_rewards.append(exp_r)

hist_probs   = np.array(hist_probs)
hist_rewards = np.array(hist_rewards)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

# Left: probability of each response type over training
ax = axes[0]
cols = ['#e74c3c','#f39c12','#2ecc71','#3498db','#9b59b6']
for i, (name, col) in enumerate(zip(response_names, cols)):
    ax.plot(hist_probs[:, i], label=f'{name} (r={response_rewards[i]})',
            color=col, linewidth=2)
ax.set_xlabel('PPO step')
ax.set_ylabel('Policy probability')
ax.set_title('Policy Evolution: from Uniform → Prefers Best Responses')
ax.legend(fontsize=7.5, loc='center right')
ax.grid(True, alpha=0.3)

# Right: expected reward over training
ax = axes[1]
ax.plot(hist_rewards, color='#27ae60', linewidth=2.5)
ax.axhline(response_rewards.max(), color='gray', linestyle='--',
           linewidth=1, label=f'Max possible reward={response_rewards.max()}')
ax.set_xlabel('PPO step')
ax.set_ylabel('Expected reward')
ax.set_title('Expected Reward Increases with PPO\n(KL penalty prevents full collapse to single action)')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.suptitle('Step 3: PPO Policy Optimisation Against Reward Model',
             fontsize=11, fontweight='bold')
plt.tight_layout()
plt.show()

final_probs = hist_probs[-1]
print('Final policy probabilities:')
for name, p, r in zip(response_names, final_probs, response_rewards):
    bar = '█' * int(p * 40)
    print(f'  {name:<20} p={p:.3f} {bar}')

## Exam Quick-Reference

**Answer: C**

| MDP Component | RLHF | |
|---|---|---|
| State | User prompt | What the agent perceives |
| Action | Model-generated response | What the agent does |
| Policy | Language model | Maps state→action |
| Reward | Score from reward model (trained on human preferences) | Signal to improve |

**Why not A:** Human preferences train the reward model — they're not the action.

**Why not B:** Token-level actions don't map to the RLHF MDP; the full response is the action.

**Why not D:** Rank assigned by reward model is not an action; it's an evaluation.

**RLHF 3 steps:** SFT → Reward Model → PPO (with KL penalty against SFT to prevent reward hacking).